# Assignment A6 — First MP1 Visualization (Open Food Facts breakfast cereals)

This notebook contains visualisations for all the analytical questions defined for MP1. It uses **Plotly** on `cereals_breakfast.csv`.

**Analytical choices:** brands are used as a single string per row; Nutri-Score `unknown` rows are excluded from grade-based analyses; “C or worse” means grades **c**, **d**, **e** only; `countries` values are mapped to normalized English country names; **salt** (`salt_100g`) is used as the salinity proxy (no sodium column in this extract).


In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

CSV_PATH = Path("cereals_breakfast.csv")
df = pd.read_csv(CSV_PATH)

# Numeric nutrients (missing cells become NaN)
NUM_COLS = [
    "sugars_100g",
    "fiber_100g",
    "salt_100g",
    "saturated-fat_100g",
    "energy-kcal_100g",
]
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# --- Country normalization (raw Open Food Facts strings → canonical English) ---
_COUNTRY_RAW_TO_CANONICAL = {
    "France": "France",
    "Belgium": "Belgium",
    "Belgien": "Belgium",
    "Belgique": "Belgium",
    "Bélgica": "Belgium",
    "United Kingdom": "United Kingdom",
    "en:United Kingdom": "United Kingdom",
    "en:united-kingdom": "United Kingdom",
    "Royaume-Uni": "United Kingdom",
    "Vereinigtes Königreich": "United Kingdom",
    "en:gb": "United Kingdom",
    "gb": "United Kingdom",
    "Spain": "Spain",
    "España": "Spain",
    "Ireland": "Ireland",
    "Irlande": "Ireland",
    "Bulgaria": "Bulgaria",
    "Bugarska": "Bulgaria",
    "Bułgaria": "Bulgaria",
    "Bulgária": "Bulgaria",
    "Austria": "Austria",
    "Áustria": "Austria",
    "Netherlands": "Netherlands",
    "Portugal": "Portugal",
    "Italy": "Italy",
    "Italia": "Italy",
    "Itália": "Italy",
    "Jordan": "Jordan",
    "Armenia": "Armenia",
    "Francia": "France",
    "Frankrijk": "France",
    "Frankreich": "France",
    "Cyprus": "Cyprus",
    "en:france": "France",
    "en:fr": "France",
    "fr": "France",
    "Finland": "Finland",
    "Côte d'Ivoire": "Côte d'Ivoire",
    "Canada": "Canada",
    "Morocco": "Morocco",
    "en:ma": "Morocco",
    "Algérie": "Algeria",
    "Algeria": "Algeria",
    "Deutschland": "Germany",
    "Germany": "Germany",
    "Alemania": "Germany",
    "United States": "United States",
    "Hong Kong": "Hong Kong",
    "Croatia": "Croatia",
    "Hrvatska": "Croatia",
    "Kuwait": "Kuwait",
    "Mexico": "Mexico",
    "Panama": "Panama",
}


def normalize_country(raw: object) -> str:
    if pd.isna(raw) or str(raw).strip() == "":
        return "Unknown"
    key = str(raw).strip()
    return _COUNTRY_RAW_TO_CANONICAL.get(key, key)


df["country_norm"] = df["countries"].map(normalize_country)

# Nutri-Score: keep raw; ordered label for plots (exclude unknown from ordered charts)
_grade_to_order = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
df["nutrition_grade_lower"] = df["nutrition_grade_fr"].astype(str).str.lower().str.strip()
df["grade_ordered"] = df["nutrition_grade_lower"].map(_grade_to_order)
df["grade_label"] = df["nutrition_grade_lower"].map(
    lambda g: {"a": "A", "b": "B", "c": "C", "d": "D", "e": "E"}.get(g, np.nan)
)

print(f"Rows: {len(df)}")
print("Country (normalized) value counts (top 15):")
print(df["country_norm"].value_counts().head(15))



Rows: 450
Country (normalized) value counts (top 15):
country_norm
France            216
United Kingdom     76
Belgium            66
Spain              17
Ireland            14
Portugal           12
Bulgaria            8
Italy               7
Austria             6
Morocco             5
Germany             4
Finland             2
Canada              2
Algeria             2
United States       2
Name: count, dtype: int64


### **Question 1**
Among breakfast cereals in the database, which brands have the highest average sugar content per 100g, and does a product's Nutri-Score (A–E) reliably rank with its actual sugar level?

I split question into two ideas — brands and grades. 
1. The bar chart (brands on the side, average sugar on the bottom) shows which brand strings have the highest average sugar per 100 g. It only uses rows with a known Nutri-Score and brands with at least three products, and it shows the top brands by that average so very rare brands do not dominate. 
2. The box plot shows sugar for every product, grouped by Nutri-Score letter (A through E). You can see typical sugar and spread for each letter; if worse letters often go with more sugar, the boxes drift upward toward E. Nutri-Score is not based on sugar alone, so the pattern can be messy. The line chart draws the average sugar per letter in one simple curve. The printed Spearman number sums up, in one statistic, whether “worse” letters tend to pair with more sugar across products with a known grade.


In [2]:
# --- Q1: Brand average sugar vs Nutri-Score vs sugar ---
MIN_PRODUCTS_PER_BRAND = 3

df_known_grade = df[df["grade_label"].notna()].copy()
df_brand = (
    df_known_grade.groupby("brands", as_index=False)
    .agg(n=("sugars_100g", "size"), mean_sugar=("sugars_100g", "mean"))
)
df_brand = df_brand[df_brand["n"] >= MIN_PRODUCTS_PER_BRAND].sort_values(
    "mean_sugar", ascending=False
)
top_k = 18
df_top_brands = df_brand.head(top_k)

fig_brands = px.bar(
    df_top_brands,
    x="mean_sugar",
    y="brands",
    orientation="h",
    text="mean_sugar",
    title=(
        f"Mean sugars (g/100g) by brand — known Nutri-Score only, "
        f"brands with ≥{MIN_PRODUCTS_PER_BRAND} products (top {top_k} by sugar)"
    ),
    labels={"mean_sugar": "Mean sugars (g/100g)", "brands": "Brand"},
    color="mean_sugar",
    color_continuous_scale="Reds",
)
fig_brands.update_traces(texttemplate="%{text:.1f}", textposition="outside")
fig_brands.update_layout(yaxis={"categoryorder": "total ascending"}, showlegend=False)
fig_brands.show()

fig_box = px.box(
    df_known_grade,
    x="grade_label",
    y="sugars_100g",
    category_orders={"grade_label": ["A", "B", "C", "D", "E"]},
    title="Sugar content (g/100g) by Nutri-Score — excludes unknown grades",
    labels={"grade_label": "Nutri-Score", "sugars_100g": "Sugars (g/100g)"},
    color="grade_label",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig_box.update_layout(showlegend=False)
fig_box.show()

# Ordinal correlation (1=A … 5=E): higher score = worse letter in some sense;
# we expect positive correlation if worse letters align with more sugar.
sub = df_known_grade.dropna(subset=["sugars_100g", "grade_ordered"])
rho = sub["grade_ordered"].corr(sub["sugars_100g"], method="spearman")
print(f"Spearman rho(grade ordinal vs sugars_100g): {rho:.3f} (n={len(sub)})")

grade_means = (
    df_known_grade.dropna(subset=["sugars_100g"])
    .groupby("grade_label", as_index=False)["sugars_100g"]
    .mean()
    .sort_values("grade_label", key=lambda s: s.map({"A": 1, "B": 2, "C": 3, "D": 4, "E": 5}))
)
fig_mean_line = px.line(
    grade_means,
    x="grade_label",
    y="sugars_100g",
    markers=True,
    title="Mean sugar by Nutri-Score (summary trend)",
    labels={"grade_label": "Nutri-Score", "sugars_100g": "Mean sugars (g/100g)"},
)
fig_mean_line.update_xaxes(categoryorder="array", categoryarray=["A", "B", "C", "D", "E"])
fig_mean_line.show()



Spearman rho(grade ordinal vs sugars_100g): 0.730 (n=434)


### **Question 2**
What share of products that include health-oriented keywords ("whole grain," "natural," "fiber") in their product name have a Nutri-Score of C or worse?

The code first finds rows whose product name matches those keywords. It then keeps only rows with a known A–E score (unknown is dropped). C or worse means C, D, or E only. The donut chart shows two slices: how many of those products are C–E versus A–B, as percent, label, and count on the chart. The printed lines below repeat the counts and percentages so you can quote them easily; with a small number of matching products, the share can swing a lot, which is worth saying in a report.


In [3]:
# --- Q2: Health keywords in product name vs Nutri-Score C or worse ---
_kw_pat = re.compile(
    r"whole[\s-]?grain|wholegrain|\bnatural\b|fiber|fibre",
    re.IGNORECASE,
)
df["health_keywords_name"] = df["product_name"].astype(str).str.contains(
    _kw_pat, regex=True, na=False
)

# Among keyword hits with known A–E grade only
mask_kw = df["health_keywords_name"] & df["grade_label"].notna()
df_kw = df.loc[mask_kw].copy()
n_kw = len(df_kw)
n_c_or_worse = df_kw["nutrition_grade_lower"].isin(["c", "d", "e"]).sum()
n_better = df_kw["nutrition_grade_lower"].isin(["a", "b"]).sum()

pie_df = pd.DataFrame(
    {
        "segment": ["Nutri-Score C, D, or E", "Nutri-Score A or B"],
        "count": [n_c_or_worse, n_better],
    }
)
pie_df = pie_df[pie_df["count"] > 0]

fig_kw = px.pie(
    pie_df,
    names="segment",
    values="count",
    title=(
        "Among products whose name mentions whole grain / natural / fiber: "
        "share with Nutri-Score C–E vs A–B (unknown grades excluded)"
    ),
    hole=0.45,
    color="segment",
    color_discrete_map={
        "Nutri-Score C, D, or E": "#c0392b",
        "Nutri-Score A or B": "#27ae60",
    },
)
fig_kw.update_traces(textposition="inside", textinfo="percent+label+value")
fig_kw.show()

print(
    f"Products with keyword(s) in name and known grade: {n_kw}\n"
    f"  C–E: {n_c_or_worse} ({100 * n_c_or_worse / n_kw:.1f}%)\n"
    f"  A–B: {n_better} ({100 * n_better / n_kw:.1f}%)"
)



Products with keyword(s) in name and known grade: 10
  C–E: 5 (50.0%)
  A–B: 5 (50.0%)


### **Question 3**
Which nutritional fields (e.g., fiber, sodium, saturated fat, energy) are most frequently missing across products in this category, and does data completeness vary by country of origin?

The first bar chart looks at the whole dataset and, for each nutrient column (sugar, fiber, salt, saturated fat, energy per 100 g), shows what share of products has a blank value. Longer bars mean that field is missing more often, so you see which numbers are least complete overall. The heatmap uses normalized country names (so different spellings and codes for the same country are grouped together) and colors how often each field is missing inside each country. Only countries with enough products are included so one-off rows do not define a whole country. Darker cells mean more gaps for that country and that field, which supports comparing data completeness by place as well as which fields are weakest overall.

In [4]:
# --- Q3: Missing nutritional fields; completeness by normalized country ---
MISSING_COLS = NUM_COLS
overall_pct = (
    pd.DataFrame(
        {
            "field": MISSING_COLS,
            "pct_missing": [100 * df[c].isna().mean() for c in MISSING_COLS],
        }
    )
    .sort_values("pct_missing", ascending=True)
)

fig_miss = px.bar(
    overall_pct,
    x="pct_missing",
    y="field",
    orientation="h",
    title="Share of products with missing values (per 100g field)",
    labels={"pct_missing": "% missing", "field": "Field"},
    color="pct_missing",
    color_continuous_scale="Blues",
)
fig_miss.update_layout(showlegend=False)
fig_miss.show()

MIN_PER_COUNTRY = 8
counts = df["country_norm"].value_counts()
keep_countries = counts[counts >= MIN_PER_COUNTRY].index.tolist()
df_ct = df[df["country_norm"].isin(keep_countries)].copy()

miss_by = []
for country in sorted(df_ct["country_norm"].unique()):
    sub = df_ct[df_ct["country_norm"] == country]
    for col in MISSING_COLS:
        miss_by.append(
            {
                "country_norm": country,
                "field": col,
                "pct_missing": 100 * sub[col].isna().mean(),
                "n": len(sub),
            }
        )
heat_df = pd.DataFrame(miss_by)

fig_heat = px.imshow(
    heat_df.pivot(index="country_norm", columns="field", values="pct_missing"),
    labels=dict(x="Field", y="Country (normalized)", color="% missing"),
    title=(
        f"Missing data rate (%) by country — countries with ≥{MIN_PER_COUNTRY} products"
    ),
    color_continuous_scale="YlOrRd",
    aspect="auto",
)
fig_heat.update_layout(xaxis={"side": "bottom"})
fig_heat.show()

